In [1]:
# ==========================================
# Import Libraries
# ==========================================

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
#Load Dataset

df = pd.read_csv("../datasets/raw/superstore.csv", encoding="latin1")

df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

df["Month"] = df["Order Date"].dt.to_period("M").astype(str)

In [ ]:
# ==========================================
# Business Problem Analysis
# ==========================================

In [ ]:
# High Sales but Low / Negative Profit

product_performance = (
    df.groupby("Sub-Category")
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum")
      )
      .sort_values("Sales", ascending=False)
)

product_performance

##Main business insight: High sales don't necessarily mean high profitability. 
# Tables are the biggest red flag and should be investigated further, especially for discounting/cost factors.

,Sales,Profit
Sub-Category,,
Phones,330007.0540,44515.7306
Chairs,328449.1030,26590.1663
Storage,223843.6080,21278.8264
Tables,206965.5320,-17725.4811
Binders,203412.7330,30221.7633
Machines,189238.6310,3384.7569
Accessories,167380.3180,41936.6357
Copiers,149528.0300,55617.8249
Bookcases,114879.9963,-3472.5560


In [ ]:
# Profit Margin Analysis

product_performance["Profit Margin %"] = (
    product_performance["Profit"] /
    product_performance["Sales"] * 100
).round(2)

product_performance

### Main insight: Copiers has the highest profit margin (37.2%), 
# while Tables has a negative margin (-8.56%), indicating a major profitability issue despite strong sales.
#Machines has decent sales but very low profit margin (1.79%), indicating weak profitability.

,Sales,Profit,Profit Margin %
Sub-Category,,,
Phones,330007.0540,44515.7306,13.49
Chairs,328449.1030,26590.1663,8.10
Storage,223843.6080,21278.8264,9.51
Tables,206965.5320,-17725.4811,-8.56
Binders,203412.7330,30221.7633,14.86
Machines,189238.6310,3384.7569,1.79
Accessories,167380.3180,41936.6357,25.05
Copiers,149528.0300,55617.8249,37.20
Bookcases,114879.9963,-3472.5560,-3.02


In [ ]:
# ==========================================
# Discount Analysis - Tables
# ==========================================

tables = df[df["Sub-Category"] == "Tables"]

tables_discount = (
    tables.groupby("Discount")
          .agg(
              Sales=("Sales", "sum"),
              Profit=("Profit", "sum")
          )
          .sort_index()
)

tables_discount

###Main insight: 
# Tables are profitable at 0% discount, but losses increase significantly as discounts rise. 
# At 40% discount, the loss is highest (~$16.2K), indicating that heavy discounting is a major driver of Tables' negative profitability.

###Recommendation: 
# Reduce high discounts on Tables and review pricing/margin before offering heavy promotions.

,Sales,Profit
Discount,,
0.00,71578.760,13276.2997
0.20,45430.232,-303.5580
0.30,25182.150,-3402.3276
0.40,45614.406,-16187.3968
0.45,5484.974,-2493.1111
0.50,13675.010,-8615.3873


In [ ]:
# Profit Margin Analysis
tables_discount["Profit Margin %"] = (
    tables_discount["Profit"] /
    tables_discount["Sales"] * 100
).round(2)

tables_discount

##Main insight: 
# Profit margin drops sharply as discount increases. 
# Tables have an 18.55% margin at 0% discount, but it falls to -63% at 50% discount, showing that heavy discounting significantly reduces profitability.

##Recommendation: 
# Avoid heavy discounts on Tables and review pricing before running high-discount promotions.

,Sales,Profit,Profit Margin %
Discount,,,
0.00,71578.760,13276.2997,18.55
0.20,45430.232,-303.5580,-0.67
0.30,25182.150,-3402.3276,-13.51
0.40,45614.406,-16187.3968,-35.49
0.45,5484.974,-2493.1111,-45.45
0.50,13675.010,-8615.3873,-63.00


In [ ]:
# ==========================================
# Product Portfolio Analysis
# ==========================================

product_performance["Sales Rank"] = (
    product_performance["Sales"].rank(ascending=False)
)

product_performance["Profit Rank"] = (
    product_performance["Profit"].rank(ascending=False)
)

product_performance

##Main insight: 
# Phones are a strong high-sales, high-profit performer, 
# while Tables generate high sales but have the lowest profitability, making them a key product to optimize.

,Sales,Profit,Profit Margin %,Sales Rank,Profit Rank
Sub-Category,,,,,
Phones,330007.0540,44515.7306,13.49,1.0,2.0
Chairs,328449.1030,26590.1663,8.10,2.0,6.0
Storage,223843.6080,21278.8264,9.51,3.0,7.0
Tables,206965.5320,-17725.4811,-8.56,4.0,17.0
Binders,203412.7330,30221.7633,14.86,5.0,5.0
Machines,189238.6310,3384.7569,1.79,6.0,13.0
Accessories,167380.3180,41936.6357,25.05,7.0,3.0
Copiers,149528.0300,55617.8249,37.20,8.0,1.0
Bookcases,114879.9963,-3472.5560,-3.02,9.0,16.0


In [ ]:
# ==========================================
# Regional Sub-Category Profitability
# ==========================================

region_subcat_profit = (
    df.groupby(["Region", "Sub-Category"])["Profit"]
      .sum()
      .unstack()
)

region_subcat_profit

##Main insight: 
# Tables are unprofitable in Central, East, and South regions, with the East region showing the highest loss (~$11K). 
# However, Tables remain profitable in the West region.

##Recommendation: 
# Investigate pricing and discounting for Tables in the East region, where the profitability issue is most severe.

Sub-Category,Accessories,Appliances,Art,Binders,Bookcases,Chairs,Copiers,Envelopes,Fasteners,Furnishings,Labels,Machines,Paper,Phones,Storage,Supplies,Tables
Region,,,,,,,,,,,,,,,,,
Central,7251.6306,-2638.6175,1195.1591,-1043.6369,-1997.9043,6592.7221,15608.8413,1777.5283,236.6186,-3906.2168,1073.0794,-1486.0666,6971.9005,12323.0267,1969.8365,-661.8881,-3559.6504
East,11195.8644,8391.4134,1899.9443,11267.9346,-1167.6318,9357.7706,17022.8418,1812.4090,263.9908,5881.4071,1129.2800,6928.6429,9015.3710,12314.6860,8389.3712,-1155.1352,-11025.3801
South,7004.5424,4123.9396,1058.5866,3900.6640,1339.4918,6612.0893,3658.9067,1465.4770,173.7181,3442.6829,1040.7723,-1438.8930,5947.0614,10767.2753,2274.2965,1.8773,-4623.0579
West,16484.5983,8261.2699,2374.0970,16096.8016,-1646.5117,4027.5843,19327.2351,1908.7624,275.1907,7641.2704,2303.1223,-618.9264,12119.2364,9110.7426,8645.3222,626.0465,1482.6073


In [ ]:
#Q>Are high discounts responsible for Tables losses in the East region?
# ==========================================
# East Region - Tables Discount Analysis
# ==========================================

east_tables = df[
    (df["Region"] == "East") &
    (df["Sub-Category"] == "Tables")
]

east_tables_discount = (
    east_tables.groupby("Discount")
               .agg(
                   Sales=("Sales", "sum"),
                   Profit=("Profit", "sum")
               )
               .sort_index()
)

east_tables_discount

##Main insight: 
# In the East region, Tables generate losses at both 30% and 40% discounts, 
# with losses increasing sharply at 40% discount (~$9.8K).

##Recommendation: 
# Avoid 40% discounts on Tables in the East region and 
# review pricing before offering high discounts.

,Sales,Profit
Discount,,
0.3,9421.489,-1185.6510
0.4,29718.318,-9839.7291


In [ ]:
#Q>Which customer segments are driving sales and profitability?
# ==========================================
# Segment Profitability Analysis
# ==========================================

segment_performance = (
    df.groupby("Segment")
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum")
      )
      .sort_values("Profit", ascending=False)
)

segment_performance

##Main insight: 
# Consumer is the largest segment by both Sales (~$1.16M) and Profit (~$134K), 
# while Home Office has the lowest Sales and Profit.

,Sales,Profit
Segment,,
Consumer,1.161401e+06,134119.2092
Corporate,7.061464e+05,91979.1340
Home Office,4.296531e+05,60298.6785


In [ ]:
# Profit Margin Analysis
segment_performance["Profit Margin %"]=(
    segment_performance["Profit"]/
    segment_performance["Sales"]*100
).round(2)

segment_performance

##Main insight: 
# Home Office has the highest profit margin (14.03%), 
# while Consumer generates the highest sales and profit but has the lowest margin (11.55%).

,Sales,Profit,Profit Margin %
Segment,,,
Consumer,1.161401e+06,134119.2092,11.55
Corporate,7.061464e+05,91979.1340,13.03
Home Office,4.296531e+05,60298.6785,14.03


In [ ]:
#Q>Which product categories are most/least profitable within each customer segment?

# ==========================================
# Segment-wise Category Profitability
# ==========================================

segment_category_profit = (
    df.groupby(["Segment", "Category"])["Profit"]
      .sum()
      .unstack()
)

segment_category_profit

##Main insight: 
# Technology is the most profitable category across all customer segments, 
# while Furniture generates the lowest profit.

Category,Furniture,Office Supplies,Technology
Segment,,,
Consumer,6991.0786,56330.3210,70797.8096
Corporate,7584.8158,40227.3202,44166.9980
Home Office,3875.3784,25933.1596,30490.1405


In [ ]:
#Q>Which categories are underperforming in specific regions?

# ==========================================
# Category-wise Regional Profitability
# ==========================================

category_region_profit = (
    df.groupby(["Region", "Category"])["Profit"]
      .sum()
      .unstack()
)

category_region_profit

##Main insight: 
# Furniture is the weakest category, generating a loss in the Central region (-$2.9K), 
# while Technology is the strongest category across all regions.

##Recommendation: 
# Investigate Furniture profitability in the Central region, especially pricing and discounting.

Category,Furniture,Office Supplies,Technology
Region,,,
Central,-2871.0494,8879.9799,33697.4320
East,3046.1658,41014.5791,47462.0351
South,6771.2061,19986.3928,19991.8314
West,11504.9503,52609.8490,44303.6496


In [ ]:
# ==========================================
# Final Business KPI Summary
# ==========================================

total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
profit_margin = (total_profit / total_sales) * 100
total_orders = df["Order ID"].nunique()
avg_discount = df["Discount"].mean() * 100

kpi_summary = {
    "Total Sales": round(total_sales, 2),
    "Total Profit": round(total_profit, 2),
    "Profit Margin %": round(profit_margin, 2),
    "Total Orders": total_orders,
    "Average Discount %": round(avg_discount, 2)
}

kpi_summary

##Main insight: 
# The business generated $2.30M in sales with $286.4K profit and 
# a 12.47% overall profit margin across 5,009 orders.

{'Total Sales': np.float64(2297200.86),
 'Total Profit': np.float64(286397.02),
 'Profit Margin %': np.float64(12.47),
 'Total Orders': 5009,
 'Average Discount %': np.float64(15.62)}

In [ ]:
# ==========================================
# Final Business Recommendations
# ==========================================

#1. Reduce heavy discounts on Tables, especially in the East region, where losses are highest.

#2. Review pricing and margins for low-profit products such as Tables, Bookcases, and Machines.

#3. Focus on high-profit products such as Copiers and Phones to improve overall profitability.

#4. Investigate Furniture performance in the Central region, where it generates a loss.

#5. Maintain strong focus on Technology, which consistently generates high profit across regions and customer segments.


In [ ]:
# ==========================================
# Project Conclusion
# ==========================================

# The analysis identified key profitability gaps across products, regions, and customer segments. 
# The findings show that high sales do not always translate into high profitability, with discounting 
# being a major concern for Tables. These insights can help management improve pricing, discount 
# strategies, and product-level profitability.